In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

import requests

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

import numpy as np

import shutil

from time import sleep

import os

import re

from selenium.webdriver.chrome.service import Service as ChromeService

from webdriver_manager.chrome import ChromeDriverManager

import pdfplumber
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)




In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ID OJK' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

#writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running ID OJK Web Scraping Tool v.1.3


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [4]:
regdict={
          'ID OJK 1': 'https://www.ojk.go.id/id/kanal/perbankan/data-dan-statistik/Pages/Daftar-Alamat-Kantor-Pusat-Bank-Umum-Dan-Syariah.aspx',    

          'ID OJK 2': 'https://www.ojk.go.id/id/kanal/perbankan/data-dan-statistik/Pages/Daftar-Alamat-Kantor-Perwakilan-Bank-di-Luar-Negeri.aspx', 

          'ID OJK 3': 'https://www.ojk.go.id/id/kanal/perbankan/data-dan-statistik/Pages/Daftar-Alamat-Kantor-Pusat-BPR.aspx', 

          'ID OJK 4': 'https://www.ojk.go.id/id/kanal/perbankan/data-dan-statistik/Pages/Daftar-Alamat-Kantor-Pusat-BPRS.aspx', 

          'ID OJK 5': 'https://www.ojk.go.id/id/kanal/pasar-modal/data-dan-statistik/data-perusahaan-efek/Default.aspx', 
        # multiple sheets, keep all or only first one? 
         'ID OJK 6': 'https://www.ojk.go.id/id/Fungsi-Utama/Pasar-Modal/Informasi-Pasar-Modal/Daftar-Pelaku-Pasar-Modal-Indonesia/Default.aspx', 
        # is individual captical market player? click into also has 4 subfolders

         'ID OJK 7': 'https://www.ojk.go.id/id/kanal/iknb/data-dan-statistik/direktori/asuransi/Default.aspx', 
        
         'ID OJK 8': 'https://www.ojk.go.id/id/Fungsi-Utama/Perasuransian-Penjaminan-Dana-Pensiun/Informasi-PPDP/Direktori-Lembaga-Penjaminan/Default.aspx', 
        
         'ID OJK 9': 'https://www.ojk.go.id/id/kanal/iknb/data-dan-statistik/direktori/dana-pensiun/Default.aspx',
        # 7/8/9 similar visit logic, can reuse
         'ID OJK 10': 'https://www.ojk.go.id/id/Fungsi-Utama/Perasuransian-Penjaminan-Dana-Pensiun/Informasi-PPDP/Direktori-Jasa-Penunjang-Asuransi/Default.aspx',
        # individual, some excel embeded

         'ID OJK 11': 'https://www.ojk.go.id/id/kanal/iknb/data-dan-statistik/direktori/lembaga-pembiayaan/Default.aspx', 
        # 11 is PDF download, different from others

         'ID OJK 12': 'https://www.ojk.go.id/id/kanal/iknb/data-dan-statistik/direktori/direktori-lkm/Default.aspx', 
        # 12 is PDF download, different from others

          'ID OJK 13': 'https://www.ojk.go.id/id/Fungsi-Utama/ITSK/Perizinan-ITSK-Aset-Keuangan-Digital-Aset-Kripto/Default.aspx', 
        # 13 is PDF download, different from others

        }

Typology ={

        'ID OJK 1': 'List of Commercial Banks and Sharia Banks',    

        'ID OJK 2': 'List of Representative Offices of Foreign Banks in Indonesia', 

        'ID OJK 3': 'List of Rural Bank (BPR)', 

        'ID OJK 4': 'List of Sharia Rural Banks (BPRS)', 

        'ID OJK 5': 'List of Securities Company', 

        'ID OJK 6': 'List of Capital Market Players in Indonesia', 
        
        'ID OJK 7': 'List of Insurance Companies', 
        
        'ID OJK 8': 'List of Guarantee Companies', 

        'ID OJK 9': 'List of Pension Funds',

        'ID OJK 10': 'List of Insurance Brokerage Companies, Reinsurance Brokerage Companies, and Insurance Loss Adjusters',
        'ID OJK 11': 'List of Financing Institutions', 

        'ID OJK 12': 'List of Microfinance Institutions', 

        'ID OJK 13': 'List of Digital Financial Asset Trading Providers', 



        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')



city_list = [
"Jakarta", "Surabaya", "Bekasi", "Bandung", "Medan", "Depok", "Tangerang",
"Palembang", "Semarang", "Makassar", "South Tangerang", "Batam", "Bogor",
"Pekanbaru", "Bandar Lampung", "Padang", "Samarinda", "Malang",
"Tasikmalaya", "Serang", "Balikpapan", "Banjarmasin", "Pontianak",
"Denpasar", "Jambi", "Cimahi", "Surakarta", "Nusantara", "Kupang",
"Manado", "Cilegon", "Mataram", "Jayapura", "Bengkulu", "Palu",
"Yogyakarta", "Sukabumi", "Ambon", "Kendari", "Cirebon", "Dumai",
"Pekalongan", "Palangka Raya", "Binjai", "Kediri", "Sorong",
"Pematangsiantar", "Banjarbaru", "Tegal", "Banda Aceh", "Tarakan",
"Probolinggo", "Singkawang", "Lubuk Linggau", "Padang Sidempuan",
"Tanjungpinang", "Bitung", "Pangkalpinang", "Batu", "Pasuruan", "Banjar",
"Gorontalo", "Ternate", "Madiun", "Salatiga", "Prabumulih",
"Lhokseumawe", "Langsa", "Bontang", "Tanjungbalai", "Tebing Tinggi",
"Metro", "Palopo", "Bima", "Baubau", "Parepare", "Blitar", "Pagar Alam",
"Payakumbuh", "Gunungsitoli", "Mojokerto", "Bukittinggi", "Kotamobagu",
"Magelang", "Tidore Islands", "Tomohon", "Sungai Penuh", "Subulussalam",
"Pariaman", "Sibolga", "Tual", "Solok", "Sawahlunto", "Padang Panjang",
"Sabang",
]


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def extract_city(address: str, cities=city_list):
    text = address.lower()
    for city in cities:  # longest first
        if re.search(rf"\b{re.escape(city.lower())}\b", text):
            return city
    return ''

zip_pattern = re.compile(r"\b\d{5}\b")

def extract_zip(address: str):
    """Return the first 5-digit ZIP in the address, or None if none found."""
    match = zip_pattern.search(address)
    return match.group(0) if match else ''

def pick_col(df, keys):
    """First column whose name contains any key (case-insensitive); else None."""
    for col in df.columns:
        col_l = str(col).lower()
        for k in keys:
            if str(k).lower() in col_l:
                return col
    return None


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
# TO DO
# Check List 5 and List 6; List 5 contain revokje company, remove them. List 6 missing a sheet

for index, reg in enumerate(regdict):
    print(f'[Start New Reg] -- Working with list { reg} --')
    try:
        resp =  requests.get(regdict[reg],verify=False)  # to avoid timeout issues
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        status = getattr(e.response, "status_code", "no-response")
        raise RuntimeError(
            f"[ERROR] -- Status: {status} Unable to access the URL: {regdict[reg]} (reason: {e}) --"
        ) from e
    soup = BeautifulSoup(resp.content, 'html.parser')
    if reg == regulatorName+' 1':
        links = soup.find_all('a')
        for link in links:
            text = (link.get_text() or '').strip()
            href = (link.get('href') or '').strip()
            if text.lower().endswith('.xlsx') or href.lower().endswith('.xlsx'):
                driver.get('https://www.ojk.go.id'+href)

        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

        if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
            sleep(3)
            print('[INFO] -- Check the download file --')

        else:
            print('[INFO] -- Maybe the file link is error, Change to xlsx -- ')
            driver.get(regdict[reg][:]+'x')
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]


        with pd.ExcelFile(dl_files[0]) as xlsx:
            print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
            print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")
            # Load the first sheet into a DataFrame
            data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
        
            data = data[data.iloc[:, 0].notna()]
            tags_ = data.iloc[:, 0]
            name_ = data.iloc[:, 1]
            address_ = data.iloc[:, 2]
            phone_ = data.iloc[:, 3]
            website_ = data.iloc[:, 4]
            for idx, tag_ in tags_.items():
                if isinstance(tag_, (int, np.integer)):
                    each_address = address_.loc[idx].replace('\n',' ').strip()
                    # print(extract_city(each_address))
                    each_zip = extract_zip(each_address)
                    # print(each_zip)
                    eacn_phone = phone_.loc[idx]
                    if len(str(eacn_phone))>3:
                        eacn_phone = str(eacn_phone).replace('\n',' ').strip()
                    else:
                        eacn_phone = ''
                    # print(eacn_phone)
                    # print(website_.loc[idx])
                    # print(sub_type)
                    sqldict['Name'].append(name_.loc[idx].replace('\n',' ').strip())
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Address_1'].append(each_address)
                    sqldict['City'].append(extract_city(each_address))
                    sqldict['Zip'].append(each_zip)
                    sqldict['Phone'].append(eacn_phone)
                    sqldict['Website'].append(website_.loc[idx])
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict["Cntry"].append("ID")  
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
                else:
                    #print(f'[INFO] -- Subtype: {tag_} --')
                    sub_type = str(tag_).strip()              
    elif reg == regulatorName+' 2':
        tables = soup.find_all("table")

        all_data = []

        for table in tables:
            headers = [th.get_text(strip=True) for th in table.find_all("th")]
            
            # Extract rows
            rows = []
            for tr in table.find_all("tr")[1:]:  # skip header row
                cells = [td.get_text(strip=True).replace("\u200b", "") for td in tr.find_all("td")]
                if cells:
                    row_dict = dict(zip(headers, cells))
                    rows.append(row_dict)
            
            all_data.extend(rows)
        # Print all variables row by row
        for row in all_data:
            if (len(row)) >1:
                name_ = row['Nama'].lstrip()
                address_ = row['Alamat'].lstrip() 
                tele_ = row['Telepon'].lstrip()
                if name_ == '':
                    pass
                else:
                    #print(cntry, name_, address_, tele_)
                    sqldict['Name'].append(name_)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Address_1'].append(address_)
                    sqldict['City'].append(extract_city(address_))
                    sqldict['Zip'].append(extract_zip(address_))
                    sqldict['Phone'].append(tele_)
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])    
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict["Cntry"].append("ID")   
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
            else:
                cntry = row['No.']
    elif reg == regulatorName+' 3' or reg == regulatorName+' 4': 
        links = soup.find_all('a')
        for link in links:
            text = (link.get_text() or '').strip()
            href = (link.get('href') or '').strip()
            if text.lower().endswith('.xlsx') or href.lower().endswith('.xlsx'):
                driver.get('https://www.ojk.go.id'+href)

        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

        if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
            sleep(3)
            print('[INFO] -- Check the download file --')

        else:
            print('[INFO] -- Maybe the file link is error, waiting 30s -- ')
            sleep(30)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        with pd.ExcelFile(dl_files[0]) as xlsx:
            print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
            print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")
            data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
            data = data[data.iloc[:, 0].notna()]
            sandi_    = data.iloc[:, 0]
            name_ = data.iloc[:, 2]
            address_ = data.iloc[:, 3]
            city_ = data.iloc[:, 4]
            phone_ = data.iloc[:, -2]
            website_ = data.iloc[:, -1]

            for sandi, name,address,city,phone,website in zip(sandi_, name_, address_,city_,phone_,website_):
                if sandi != 'Sandi':
                    name_each = name.replace('\n',' ').strip()
                    address_each = address.replace('\n',' ').strip()
                    city_each = city.replace('\n',' ').strip()
                    tele_each = str(phone).replace('\n',' ').strip()
                    try:
                        web_each= website.replace('\n',' ').strip()
                    except:
                        web_each = ""
                    #print(name_each, address_each, city_each, tele_each, web_each)
                    sqldict['Name'].append(name_each)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Address_1'].append(address_each)
                    sqldict['City'].append(city_each)
                    sqldict['Phone'].append(tele_each)
                    sqldict['Website'].append(web_each)
                    sqldict['RegCtry'].append(reg.split(' ')[0])    
                    sqldict['RegCode'].append(reg.split(' ')[1])    
                    sqldict['ListCode'].append(reg.split(' ')[-1])  
                    sqldict["Cntry"].append("ID")   
                    sqldict['ListName'].append(Typology[reg])   
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
    elif reg in [regulatorName+' 5',  regulatorName+' 7',regulatorName+' 8', regulatorName+' 9']:
        url = 'https://www.ojk.go.id' + soup.find('div', class_='content').find('ul').find_all('a')[0]['href']
        driver.set_page_load_timeout(180)
        driver.get(url)
        sleep(3)
        wait = WebDriverWait(driver, 5)  # adjust timeout if needed
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'download-counter-0')))
        wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'download-counter-0')))

        pdf_button = driver.find_element(By.CLASS_NAME, 'download-counter-0')
        sleep(2)
        pdf_button.click()
        sleep(60)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                print('[INFO] -- Check the download file --')
                sleep(10)

        else:
                print('[INFO] -- Maybe the file link is download -- ')
                sleep(120)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

        if reg == regulatorName +' 5':
            FIELD_MAP = {
                "Name": ["nama", "nama perusahaan", "nama_perusahaan"],
                "Address": ["alamat", "alamat kantor", "address"],
                "City": ["kota", "kabupaten", "city"],
                "Zip": ["kodepos", "kode pos", "zip"],
                "Phone": ["telp", "telepon", "telpon", "phone"],
                "Fax": ["fax", "faks"],
                "Email": ["email", "e-mail"],
                "Website": ["website", "web", "situs"],
                "Typology": ["jenis", "kategori", "jenis usaha"],
                "InternalID": ["no izin", "nomor izin", "nomor_izin", "no. izin"],
                "regulated_date" : ["tgl izin usaha"]
            }
            with pd.ExcelFile(dl_files[0]) as xlsx:
                sheets = xlsx.sheet_names
                frames = []
                for s in sheets:
                    df = pd.read_excel(xlsx, sheet_name=s)

                    stop_at = None
                    first_col = df.columns[0]
                    for i, val in enumerate(df[first_col]):
                        if "cabut" in str(val).lower():
                            stop_at = i
                            break

                    if stop_at is not None:
                        df = df.iloc[:stop_at]  # keep rows before "cabut" (drop the cabut row and below)
                    row = {}
                    for canon, aliases in FIELD_MAP.items():
                        col = pick_col(df, aliases)
                        row[canon] = df[col] if col else pd.NA  # fill missing with NA

                    norm_df = pd.DataFrame(row)
                    frames.append(norm_df)

            combined = pd.concat(frames, ignore_index=True)
            combined = combined.dropna(subset=combined.columns[:2])
            for _, row in combined.iterrows():
                name_ = str(row.get("Name") or "").strip()
                address_ = str(row.get("Address") or "").strip()
                internal_id_ = row.get("InternalID")
                internal_id_clean =  "" if pd.isna(internal_id_) else str(internal_id_).strip()
                city_ = str(row.get("City") or "").strip()
                zip_ = str(row.get("Zip") or "").strip()
                orig_phone_field = str(row.get("Phone") or "").strip()
                phone_ = orig_phone_field.split('\n')[0].replace('Telp','').split('Fax.')[0].strip(" ,.:;-'\"").lstrip()
                fax_ = orig_phone_field.replace(phone_, '').strip()
                fax_ = fax_.replace('Fax.', '').replace('Fax', '').replace('Telp','').strip(" ,.:;-'\"").lstrip()
                phone_ = phone_ if len(phone_)> 4 else ''
                fax_ = fax_ if len(fax_)> 4 else ''
                val = row.get("regulated_date")
                regulated_date = "" if pd.isna(val) else str(val).strip()
                sqldict['Name'].append(name_)
                sqldict['Address_1'].append(address_)
                sqldict['InternalID_1'].append(internal_id_clean)
                sqldict['InternalID_1_type'].append('Nomor Izin Usaha')
                sqldict['City'].append(city_)
                sqldict['Zip'].append(zip_)    
                sqldict['Phone'].append(phone_)    
                sqldict['Fax'].append(fax_)  
                sqldict['RegulationDate'].append(regulated_date)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("ID")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)



        elif reg ==  regulatorName +' 7':
            sleep(3)
            with pd.ExcelFile(dl_files[0]) as xlsx:
                sheets = [s for s in xlsx.sheet_names if "asuransi" in s.lower()]
                frames = [pd.read_excel(xlsx, sheet_name=s) for s in sheets]

            combined = pd.concat(frames, ignore_index=True)
            print(f"Loaded sheets: {sheets}")
            for _,item_ in combined.iterrows():
                name_ = item_[combined.columns[1]]
                number_izin = item_[combined.columns[4]]
                date_register = item_[combined.columns[5]]
                address_ = item_[combined.columns[6]]
                city_ = item_[combined.columns[7]]
                zip_ = item_[combined.columns[8]]
                phone_ = item_[combined.columns[9]]
                email_ = item_[combined.columns[10]]
                website_ = item_[combined.columns[11]]
                sqldict['Name'].append(name_)
                sqldict['Address_1'].append(address_)
                sqldict['RegulationDate'].append(date_register)
                sqldict['Zip'].append(zip_ if len(str(zip_)) >= 2 else "")
                sqldict['City'].append(city_)
                sqldict['InternalID_1'].append(number_izin)
                sqldict['InternalID_1_type'].append(combined.columns[4])
                sqldict['ListProcessDate'].append(processdate)
                sqldict["Phone"].append(phone_ if len(str(phone_)) >= 3 else "")
                sqldict['Email'].append(email_ if len(str(email_)) >= 3 else "")
                sqldict['Website'].append(website_ if len(str(website_)) >= 3 else "")
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("ID")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
        elif reg == regulatorName +' 8':
            with pd.ExcelFile(dl_files[0]) as xlsx:
                sheets = xlsx.sheet_names
                frames = []
                for s in sheets:
                    df = pd.read_excel(xlsx, sheet_name=s)
                    # drop rows where the first two columns are NaN (change how="any" if you want to drop when either is NaN)
                    df = df.dropna(subset=df.columns[:2])
                    frames.append(df)
            combined = pd.concat(frames, ignore_index=True)
            # print(f"Loaded sheets: {sheets}")
            combined.columns = combined.iloc[0]
            combined=combined[1:]
            for _,item_ in combined.iterrows():

                if 'JENIS' in item_[combined.columns[1]]:
                    continue
                topo_ = item_[combined.columns[1]]
                name_ = item_[combined.columns[2]]
                number_izin = item_[combined.columns[3]]
                address_ = item_[combined.columns[4]]
                city_=item_[combined.columns[5]]
                phone_ = item_[combined.columns[6]]
                fax_ = item_[combined.columns[7]]
                email_ = item_[combined.columns[9]]
                website_ = item_[combined.columns[8]]
                sqldict['Name'].append(name_)
                sqldict['Typology'].append(topo_)
                sqldict['Address_1'].append(address_)
                sqldict['Zip'].append(extract_zip(address_))
                sqldict['City'].append(city_)
                sqldict['InternalID_1'].append(number_izin)
                sqldict['InternalID_1_type'].append(combined.columns[3])
                sqldict['Phone'].append(phone_ if len(str(phone_)) >= 3 else "")
                sqldict['Fax'].append(fax_ if len(str(fax_)) >= 3 else "")
                sqldict['Website'].append(website_ if len(str(website_)) >= 3 else "")
                sqldict['Email'].append(email_ if len(str(email_)) >= 3 else "")
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("ID")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')

                sqldict = bourange_same_length_array(sqldict)
        elif reg == regulatorName +' 9':

            with pd.ExcelFile(dl_files[0]) as xlsx:
                sheets = xlsx.sheet_names
                frames = []
                for s in sheets:
                    df = pd.read_excel(xlsx, sheet_name=s)
                    # drop rows where the first two columns are NaN (change how="any" if you want to drop when either is NaN)
                    df = df.dropna(subset=df.columns[:2])
                    frames.append(df)

            combined = pd.concat(frames, ignore_index=True)

            for _,item_ in combined.iterrows():
                topo_ = item_[combined.columns[1]]
                name_ = item_[combined.columns[2]]
                number_izin = item_[combined.columns[8]]
                address_ = item_[combined.columns[3]]
                city_=item_[combined.columns[5]]
                zip_ = item_[combined.columns[6]]
                phone_ = item_[combined.columns[9]]
                email_ = item_[combined.columns[-2]]
                website_ = item_[combined.columns[-1]]
                #print(topo_,name_,number_izin,address_,city_,zip_,phone_,email_,website_)
                sqldict['Name'].append(name_)
                sqldict['Typology'].append(topo_)
                sqldict['Address_1'].append(address_)
                sqldict['Zip'].append(zip_ if len(str(zip_)) >= 2 else "")
                sqldict['City'].append(city_)
                sqldict['InternalID_1'].append(number_izin)
                sqldict['InternalID_1_type'].append(combined.columns[8])
                sqldict['Phone'].append(phone_ if len(str(phone_)) >= 3 else "")
                sqldict['Website'].append(website_ if len(str(website_)) >= 3 else "")
                sqldict['Email'].append(email_ if len(str(email_)) >= 3 else "")
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split(' ')[0])
                sqldict['RegCode'].append(reg.split(' ')[1])
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("ID")  
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')

                sqldict = bourange_same_length_array(sqldict)
                
    elif reg == regulatorName+' 6':
        driver.get(regdict[reg])
        wait = WebDriverWait(driver, 10)
        link = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='Pelaku Perorangan Pasar Modal']")))

        driver.execute_script("arguments[0].click();", link) 
        folder_ids = []
        
        for a in driver.find_elements(By.CSS_SELECTOR, "#folder-list a[id*='_listFolder']"):
            text = a.text.strip()
            if text.startswith("Wakil"):
                continue
            folder_ids.append(a.get_attribute("id"))

        for inx, fid in enumerate(folder_ids):
            link = wait.until(EC.element_to_be_clickable((By.ID, fid)))
            driver.execute_script("arguments[0].click();", link)  # triggers __doPostBack

            # wait for a file link (adjust selectors if needed)
            file_link = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a[href$='.xlsx'], a[href$='.xls']"))
            )
            href = file_link.get_attribute("href")
            print(f"[INFO] downloading {href}")
            driver.execute_script("arguments[0].click();", file_link)
            driver.back()
            sleep(3)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            sleep(5)
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                sleep(5)
                print('[INFO] -- Check the download file --')
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            else:
                sleep(20)
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if inx == 0:
                with pd.ExcelFile(dl_files[0]) as xlsx:
                        print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                        print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")
                        sheet1, sheet2, sheet3 = xlsx.sheet_names[:3]
                        # Load the first sheet into a DataFrame
                        df1 = pd.read_excel(xlsx, sheet_name=sheet1).iloc[:, [1, 2, 3, 4]]  # cols 2-5
                        df2 = pd.read_excel(xlsx, sheet_name=sheet2).iloc[:, [1, 2, 3, 4]]  # cols 2-5
                        df3 = pd.read_excel(xlsx, sheet_name=sheet3).iloc[:, [1, 2, 3, 4]]     # cols 2-4


                        new_cols = ["NAME", "ADDRESS", "INTERNAL_ID", "REGISTER_DATE"] 
                        df1.columns = new_cols
                        df2.columns = new_cols
                        combined_12 = pd.concat([df1, df2], ignore_index=True)

                        df3_aligned = pd.DataFrame({
                            combined_12.columns[0]: df3.iloc[:, 0],  # df3 col1 -> df1 col1
                            combined_12.columns[1]: df3.iloc[:, 3],           # placeholder for missing df1 col2
                            combined_12.columns[2]: df3.iloc[:, 1],  # df3 col2 -> df1 col3
                            combined_12.columns[3]: df3.iloc[:, 2],  # df3 col3 -> df1 col4
                        })
                        # Optionally concat all
                        combined_all = pd.concat([combined_12, df3_aligned], ignore_index=True, sort=False)
                        combined_all = combined_all.dropna(subset=combined_all.columns[:2])
                        skip_mask = combined_all["NAME"].astype(str).str.contains("Nama", na=False)
                        # Iterate only over rows you keep
                        for _, row in combined_all.loc[~skip_mask].iterrows():
                            name_ = row["NAME"]
                            address_ = row["ADDRESS"]
                            internal_id = row["INTERNAL_ID"]
                            register_date = row['REGISTER_DATE']

                            sqldict['Name'].append(name_)
                            sqldict['Address_1'].append(address_)
                            sqldict['Zip'].append(extract_zip(address_))
                            sqldict['City'].append(extract_city(address_))
                            sqldict['InternalID_1'].append(internal_id)
                            sqldict['InternalID_1_type'].append('Internal ID')
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegulationDate'].append(register_date)
                            sqldict['RegCtry'].append(reg.split(' ')[0])
                            sqldict['RegCode'].append(reg.split(' ')[1])
                            sqldict['ListCode'].append(reg.split(' ')[-1])
                            sqldict["Cntry"].append("ID")  
                            sqldict['ListName'].append(Typology[reg])
                            sqldict['RegulationType'].append('Regulated')
                            sqldict = bourange_same_length_array(sqldict)
            elif inx == 1:
                with pd.ExcelFile(dl_files[0]) as xlsx:
                        print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                        print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")     
                        data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                        data = data.dropna(subset=["NAMA MI"])
                        pattern = re.compile(r"(?i)\b(?:telp|telp\.?|telepon|fax|faksimili|tel)\b[:\s\-]*")
                        for name_, address_, tel_, internal_id_ in zip(data['NAMA MI'], data['ALAMAT'], data['NO TELP'], data['TAHUN IZIN MI']):
                                tel_str = str(tel_) if tel_ is not None else ""
                                cleaned_tel = pattern.sub("", tel_str).strip().strip("/")
                                sqldict['Name'].append(name_)
                                sqldict['Address_1'].append(address_)
                                sqldict['Zip'].append(extract_zip(address_))
                                sqldict['City'].append(extract_city(address_))
                                sqldict['Phone'].append(cleaned_tel)
                                sqldict['InternalID_1'].append(internal_id_)
                                sqldict['InternalID_1_type'].append('TAHUN IZIN MI')
                                sqldict['ListProcessDate'].append(processdate)
                                sqldict['RegCtry'].append(reg.split(' ')[0])
                                sqldict['RegCode'].append(reg.split(' ')[1])
                                sqldict['ListCode'].append(reg.split(' ')[-1])
                                sqldict["Cntry"].append("ID")  
                                sqldict['ListName'].append(Typology[reg])
                                sqldict['RegulationType'].append('Regulated')
                                sqldict = bourange_same_length_array(sqldict)
            elif inx ==2:
                with pd.ExcelFile(dl_files[0]) as xlsx:
                    print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                    print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")     
                    data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                    pattern = re.compile(r"(?i)\b(?:telp|telp\.?|telepon|fax|faksimili|tel)\b[:\s\-]*")
                    data = data.dropna(subset=[data.columns[0]]) 
                    for type_,name_, each_type, address_, internal_id_ in zip(data.iloc[:, 0],data.iloc[:, 1],data.iloc[:, 2],data.iloc[:, 3],data.iloc[:, 4]):
                            tel_str = str(tel_) if tel_ is not None else ""
                            cleaned_tel = pattern.sub("", tel_str).strip().strip("/")
                            #ip_ = extract_city(address_)
                            if 'Pengembalian IZin' in str(type_):
                                    break
                            if type(type_) is int:
                                    sqldict['Name'].append(name_)
                                    sqldict['Typology'].append(each_type)
                                    sqldict['Address_1'].append(address_)
                                    sqldict['Zip'].append(extract_zip(address_))
                                    sqldict['City'].append(extract_city(address_))
                                    sqldict['InternalID_1'].append(internal_id_)
                                    sqldict['InternalID_1_type'].append('Surat Keputusan')
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['RegCtry'].append(reg.split(' ')[0])
                                    sqldict['RegCode'].append(reg.split(' ')[1])
                                    sqldict['ListCode'].append(reg.split(' ')[-1])
                                    sqldict["Cntry"].append("ID")  
                                    sqldict['ListName'].append(Typology[reg])
                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)            
            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))

    elif reg ==  regulatorName+' 10':
        url = 'https://www.ojk.go.id' + soup.find('div', class_='content').find('ul').find_all('a')[0]['href']
        driver.get(url)
        wait = WebDriverWait(driver, 5)  # adjust timeout if needed
        soup = BeautifulSoup(driver.page_source,'html.parser')
        div = soup.find("div", id="div-download-counter")
        file_nums =  len(div.find_all('span',class_='download-number'))

        for idx in range(file_nums):
            btn = wait.until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, f"#div-download-counter .download-counter-{idx} input.download-counter")
                )
            )
            btn.click()
            sleep(60)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            with pd.ExcelFile(dl_files[0]) as xlsx:
                print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --") 
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                data = data.dropna(subset=[data.columns[0], data.columns[1]], how="all").reset_index(drop=True)
                data.columns = data.iloc[0]
                data = data[1:]
                for _,item_ in data.iterrows():
                    name_ = item_[data.columns[1]]
                    number_izin = item_[data.columns[2]]
                    date_register = item_[data.columns[3]]
                    address_ = item_[data.columns[4]]
                    city_ = item_[data.columns[5]]
                    zip_ = item_[data.columns[6]]
                    phone_ = item_[data.columns[7]]
                    fax_ = item_[data.columns[9]]
                    email_ = item_[data.columns[10]]
                    website_ = item_[data.columns[11]]
                    sqldict['Name'].append(name_)
                    sqldict['Address_1'].append(address_)
                    sqldict['RegulationDate'].append(date_register)
                    sqldict['Zip'].append(zip_)
                    sqldict['City'].append(city_)
                    sqldict['InternalID_1'].append(number_izin)
                    sqldict['InternalID_1_type'].append(data.columns[2])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict["Phone"].append(phone_ if len(str(phone_)) >= 3 else "")
                    sqldict['Fax'].append(fax_ if len(str(fax_)) >= 3 else "")
                    sqldict['Email'].append(email_ if len(str(email_)) >= 3 else "")
                    sqldict['Website'].append(website_ if len(str(website_)) >= 3 else "")
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict["Cntry"].append("ID")  
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)

            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))

    elif reg in [regulatorName+' 11', regulatorName+' 12', regulatorName+' 13']:
        # robustly find links inside the content div (guard against None)
        content_div = soup.find('div', class_='content')
        if reg!=regulatorName+' 13':
            if content_div:
                links = content_div.find_all('a')
            else:
                links = []

            if links:
                href = links[0].get('href', '')
                if href:
                    url = 'https://www.ojk.go.id' + href
                    driver.get(url)
                    sleep(3)
                    wait = WebDriverWait(driver, 5)  # adjust timeout if needed
                    wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'download-counter-0')))
                    wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'download-counter-0')))

                    pdf_button = driver.find_element(By.CLASS_NAME, 'download-counter-0')
                    sleep(2)
                    pdf_button.click()
                    sleep(60)
                    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
                    
                if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                    print('[INFO] -- Check the download file --')
                    sleep(60)

                else:
                    print('[INFO] -- Maybe the file link is still downloading -- ')
                    sleep(60)
                    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

                    
                if reg == regulatorName+ ' 11':
                    with pdfplumber.open(dl_files[0]) as pdf:
                        for page_no, page in enumerate(pdf.pages, start=1):
                            for tbl in page.extract_tables():
                                for t in tbl:
                                    if t[0].isdigit():
                                        name_ = t[3].replace('\n','')
                                        nomor_izin = t[6]
                                        date_regulated = t[9]
                                        address_ = t[12]
                                        province_ = t[15]
                                        topology = t[18]

                                        #print(name_,nomor_izin,date_regulated,address_,province_,topology)
                                        sqldict['Name'].append(name_)
                                        sqldict['Address_1'].append(address_)
                                        sqldict['Zip'].append(extract_zip(address_))
                                        sqldict['InternalID_1'].append(nomor_izin)
                                        sqldict['InternalID_1_type'].append('Nomor Izin')
                                        sqldict['RegulationDate'].append(date_regulated)
                                        sqldict['City'].append(province_)
                                        sqldict['Typology'].append(topology)
                                        sqldict['ListProcessDate'].append(processdate)
                                        sqldict['RegCtry'].append(reg.split(' ')[0])
                                        sqldict['RegCode'].append(reg.split(' ')[1])
                                        sqldict['ListCode'].append(reg.split(' ')[-1])
                                        sqldict["Cntry"].append("ID")  
                                        sqldict['ListName'].append(Typology[reg])
                                        sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)
                    
                elif reg == regulatorName+ ' 12':
                    with pdfplumber.open(dl_files[0]) as pdf:
                        for page_no, page in enumerate(pdf.pages, start=1):
                            for tbl in page.extract_tables():
                                for t in tbl:
                                    if t[0].isdigit():
                                        #print(t)
                                        name_ = t[1].replace('\n','')
                                        topology = t[3]
                                        nomor_izin = t[6]
                                        date_regulated = t[7]
                                        address_ = t[9]
                                        province_ = t[11]
                                        no_kantor_lkm= t[-1]
                                        #print(name_,nomor_izin,date_regulated,address_,province_,topology)
                                        sqldict['Name'].append(name_)
                                        sqldict['Address_1'].append(address_)
                                        sqldict['Zip'].append(extract_zip(address_))
                                        sqldict['InternalID_1'].append(nomor_izin)
                                        sqldict['InternalID_1_type'].append('Nomor Izin')
                                        sqldict['InternalID_2'].append(no_kantor_lkm)
                                        sqldict['InternalID_2_type'].append('No. Kantor LKM')
                                        sqldict['RegulationDate'].append(date_regulated)
                                        sqldict['City'].append(province_)
                                        sqldict['Typology'].append(topology)
                                        sqldict['ListProcessDate'].append(processdate)
                                        sqldict['RegCtry'].append(reg.split(' ')[0])
                                        sqldict['RegCode'].append(reg.split(' ')[1])
                                        sqldict['ListCode'].append(reg.split(' ')[-1])
                                        sqldict["Cntry"].append("ID")  
                                        sqldict['ListName'].append(Typology[reg])
                                        sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)

        else:
            url = 'https://www.ojk.go.id' + soup.find('div', class_='content').find('ul').find_all('a')[0]['href']
            driver.get(url)
            sleep(3)
            wait = WebDriverWait(driver, 5)  # adjust timeout if needed
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'download-counter-0')))
            wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'download-counter-0')))

            pdf_button = driver.find_element(By.CLASS_NAME, 'download-counter-0')
            sleep(2)
            pdf_button.click()
            sleep(360)
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
                    print('[INFO] -- Check the download file --')
                    sleep(10)

            else:
                    print('[INFO] -- Maybe the file link is download -- ')
                    sleep(720)
                    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            bold_titles = []
            regular_infos = []
            with pdfplumber.open(dl_files[0]) as pdf:
                for page in pdf.pages:
                    bold_title = ''
                    regular_info = ''
                    for ch in page.chars:
                        font = ch.get('fontname', '')
                        fill = ch.get('non_stroking_color')

                        is_bold = 'Montserrat-Bold' in font  # substring, ignores prefix
                        is_regular = 'Montserrat-Regular' in ch.get('fontname', '')
                        rgb = tuple(round(v, 2) for v in fill) if isinstance(fill, (list, tuple)) else ()
                        is_whiteish = rgb in ((1, 1, 1), (0.99, 0.99, 0.99)) or not rgb  # allow empty/None

                        if is_bold and is_whiteish:
                            bold_title += ch['text']
                        if is_regular and is_whiteish:
                            regular_info+=ch['text']
                    #print(bold_title)
                    #print(regular_info)
                    bold_titles.append(bold_title)
                    regular_infos.append(regular_info)
            for bold_title in bold_titles:
                for name_ in bold_title.split('PT'):
                    if name_.strip()!='':
                        name_full = 'PT' + name_
                        sqldict['Name'].append(name_full)
                        sqldict['RegCtry'].append(reg.split(' ')[0])
                        sqldict['RegCode'].append(reg.split(' ')[1])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["Cntry"].append("ID")  
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['RegulationType'].append('Regulated')
            for regular_info in regular_infos:
                for info in regular_info.split(')'):
                    if info.strip()!='':
                        internal_id = info.split('(')[0]
                        date_ = info.split('(')[-1]
                        sqldict['InternalID_1'].append(internal_id)
                        sqldict['InternalID_1_type'].append('InternalID')
                        sqldict['RegulationDate'].append(date_)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict = bourange_same_length_array(sqldict)

    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

[Start New Reg] -- Working with list ID OJK 1 --
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 2  --
[INFO] -- Number of sheets: ['Daftar Bank Umum', 'daftar nama bank di SIP']  --
[Start New Reg] -- Working with list ID OJK 2 --
[Start New Reg] -- Working with list ID OJK 3 --
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['BPR']  --
[Start New Reg] -- Working with list ID OJK 4 --
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['BPRS']  --
[Start New Reg] -- Working with list ID OJK 5 --
[INFO] -- Check the download file --
[Start New Reg] -- Working with list ID OJK 6 --
[INFO] downloading https://www.ojk.go.id/id/Fungsi-Utama/Pasar-Modal/Informasi-Pasar-Modal/Daftar-Pelaku-Pasar-Modal-Indonesia/Daftar%20Perusahaan/Pelaku%20Perorangan%20Pasar%20Modal/Agen%20Penjual%20Efek%20Reksa%20Dana/Data%20APERD.xlsx
[INFO] -- Check the download file --
[INFO] -- Number of 

Cannot set gray stroke color because /'P8' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float value
Cannot set gray stroke color because /'P11' is an invalid float value
Cannot set gray non-stroke color because /'P11' is an invalid float value
Cannot set gray stroke color because /'P14' is an invalid float value
Cannot set gray non-stroke color because /'P14' is an invalid float value
Cannot set gray stroke color because /'P8' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float value
Cannot set gray stroke color because /'P11' is an invalid float value
Cannot set gray non-stroke color because /'P11' is an invalid float value
Cannot set gray stroke color because /'P14' is an invalid float value
Cannot set gray non-stroke color because /'P14' is an invalid float value
Cannot set gray stroke color because /'P8' is an invalid float value
Cannot set gray non-stroke color because /'P8' is an invalid float valu

In [7]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 3008 values.
Key 'priority' has 3008 values.
Key 'ListLabel' has 3008 values.
Key 'Typology' has 3008 values.
Key 'EntryType' has 3008 values.
Key 'Name' has 3008 values.
Key 'InternalID_1' has 3008 values.
Key 'InternalID_1_type' has 3008 values.
Key 'InternalID_2' has 3008 values.
Key 'InternalID_2_type' has 3008 values.
Key 'InternalID_3' has 3008 values.
Key 'InternalID_3_type' has 3008 values.
Key 'CoType' has 3008 values.
Key 'License_Type' has 3008 values.
Key 'Address_1' has 3008 values.
Key 'Address_2' has 3008 values.
Key 'City' has 3008 values.
Key 'Zip' has 3008 values.
Key 'Cntry' has 3008 values.
Key 'Phone' has 3008 values.
Key 'Fax' has 3008 values.
Key 'Website' has 3008 values.
Key 'Email' has 3008 values.
Key 'RegulationType' has 3008 values.
Key 'RegulationTypeCode' has 3008 values.
Key 'RegulationDate' has 3008 values.
Key 'CancellationDate' has 3008 values.
Key 'RegCtry' has 3008 values.
Key 'RegCode' has 3008 values.
Key 'ListCode' has 3008 values

In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
# df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)

# df.to_excel(writer, 'SQL Ready', index=False)

# writer.save()

# writer.close()

driver.quit()

sleep(3)



In [9]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,PT BANK RAKYAT INDONESIA (PERSERO) Tbk,,,,,...,,,,,,,,,,
1,,,,,,PT BANK MANDIRI (PERSERO) Tbk,,,,,...,,,,,,,,,,
2,,,,,,PT BANK NEGARA INDONESIA (PERSERO) Tbk,,,,,...,,,,,,,,,,
3,,,,,,PT BANK TABUNGAN NEGARA (PERSERO) Tbk,,,,,...,,,,,,,,,,
4,,,,,,PT BANK DANAMON INDONESIA Tbk,,,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,,,,,,PT Agregasi CermatIndonesia,S-136/IK.01/2025,InternalID,,,...,,,,,,,,,,
3004,,,,,,PT Loan Market Indo,S-194/IK.01/2025,InternalID,,,...,,,,,,,,,,
3005,,,,,,PT Xfers Indonesia Teknologi,S-209/IK.01/2025,InternalID,,,...,,,,,,,,,,
3006,,,,,,PT Unicorn TechnologyIndonesia,S-237/IK.01/2025,InternalID,,,...,,,,,,,,,,


In [ ]:
combined = pd.concat([df, df1], ignore_index=True)  # rows from both, reindex 0..n-1


In [ ]:
combined2 =  pd.concat([df1, df], ignore_index=True)  # rows from both, reindex 0..n-1

In [ ]:
combined2.to_excel('ID OJK SQL Ready 2025-12-08 15.21.36.xlsx',index=False)

In [11]:
import os
import re
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

df_pdf = pd.read_excel('ID OJK SQL Ready 2026-04-02 16.18.35.xlsx')

arial_path = r"C:\Windows\Fonts\arial.ttf"
if os.path.exists(arial_path):
    pdfmetrics.registerFont(TTFont('ArialUni', arial_path))
    FONT_NAME = 'ArialUni'
else:
    FONT_NAME = 'Helvetica'
    print("WARNING: arial.ttf not found, Hungarian characters may not render correctly.")

def safe_filename(name):
    name = str(name).strip()
    return re.sub(r'[\\/:*?"<>|]+', '_', name) or "UNKNOWN"

def draw_header(c, y):
    c.setFont(FONT_NAME, 14)
    c.drawString(72, y, "Name")
    c.line(72, y - 4, 540, y - 4)
    c.setFont(FONT_NAME, 12)
    return y - 24

def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont(FONT_NAME, 12)

    x = 72
    y = draw_header(c, 740)
    max_lines_per_page = 32
    line_count = 0

    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20
        line_count += 1

        if line_count >= max_lines_per_page:
            c.showPage()
            c.setFont(FONT_NAME, 12)
            y = draw_header(c, 740)
            line_count = 0

    c.save()

os.makedirs(tempfolder, exist_ok=True)

for list_code, group in df_pdf.groupby('ListCode', dropna=False):
    code = safe_filename(list_code)
    items = group['Name'].dropna().astype(str).tolist()
    if not items:
        continue
    pdf_path = os.path.join(tempfolder, f"ID OJK SQL Ready 2026-04-02 16.18.43- {code}.pdf")
    export_list_to_pdf(items, pdf_path)
